In [ ]:
import graphite


In [ ]:
dataset = graphite.Dataset(source='washington',
                           level='line',
                           training_ratio=None,
                           validation_ratio=None,
                           test_ratio=None,
                           lazy_mode=False,
                           seed=42)
print(dataset)


In [ ]:
augmentor = graphite.Augmentor(  # elastic_transform=[0.99, 43, 0.75],
    erosion=[0.99, 5, 1],
    dilation=[0.99, 2, 1],
    #    mixup=[0.99, 0.3, 1],
    #    perspective_transform=[0.99, 0.4],
    #    salt_and_pepper=[0.99, 0.3],
    #    gaussian_blur=[0.99, 11, 1],
    #    shearing=[0.99, 30],
    scaling=[0.99, 0.05],
    rotation=[0.99, 0.5],
    translation=[0.99, 0.05, 0.05],
    seed=42)
print(augmentor)


In [ ]:
model = graphite.Model(network='flor', tokenizer=dataset.tokenizer, seed=42)
model.compile(learning_rate=1e-3, run_index=None)

print(model)


In [ ]:
train_data, train_steps = dataset.get_generator(dataset.training, batch_size=16, augmentor=augmentor)
valid_data, valid_steps = dataset.get_generator(dataset.validation, batch_size=16, augmentor=None)

model.fit(training_data=train_data,
          training_steps=train_steps,
          validation_data=valid_data,
          validation_steps=valid_steps,
          plateau_cooldown=0,
          plateau_factor=0.1,
          plateau_patience=15,
          patience=20,
          epochs=1000,
          verbose=1)


In [ ]:
test_data, test_steps = dataset.get_generator(dataset.test, batch_size=16, augmentor=None)

predictions, probabilities = model.predict(test_data=test_data,
                                           test_steps=test_steps,
                                           top_paths=10,
                                           beam_width=30,
                                           ctc_decode=True,
                                           token_decode=True,
                                           verbose=1)

print(predictions)
print(predictions.shape, probabilities.shape)
print(predictions[0][0], probabilities[0][0])


In [ ]:

metrics, samples = model.evaluate(dataset.test,
                                  predictions=predictions,
                                  prediction_samples=10,
                                  best_cumulative_paths=True)

print(metrics)
print()
print(metrics.shape, samples.shape)

# samples[0][1]
